In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from PIL import Image
import cv2
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"timm: {timm.__version__}")

Device: cpu
PyTorch: 2.10.0+cpu
timm: 1.0.25


In [2]:
# load labels
df = pd.read_csv('../data/train.csv')

# update path to match actual folder name
df['filepath'] = df['id_code'].apply(
    lambda x: f'../data/train_images/{x}.png'
)

# train/validation split — 80/20, stratified by grade
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['diagnosis']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Training samples:   {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"\nTrain class distribution:")
print(train_df['diagnosis'].value_counts().sort_index())
print(f"\nValidation class distribution:")
print(val_df['diagnosis'].value_counts().sort_index())

Training samples:   2929
Validation samples: 733

Train class distribution:
diagnosis
0    1444
1     296
2     799
3     154
4     236
Name: count, dtype: int64

Validation class distribution:
diagnosis
0    361
1     74
2    200
3     39
4     59
Name: count, dtype: int64


In [3]:
class RetinalDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def apply_clahe(self, img):
        # convert to LAB color space
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        # apply CLAHE to L channel only
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        # merge back
        lab = cv2.merge([l, a, b])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # load image
        img = cv2.imread(row['filepath'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # apply CLAHE contrast enhancement
        img = self.apply_clahe(img)
        
        # convert to PIL for torchvision transforms
        img = Image.fromarray(img)
        
        if self.transform:
            img = self.transform(img)
        
        label = int(row['diagnosis'])
        return img, label

# transforms
train_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# create datasets
train_dataset = RetinalDataset(train_df, transform=train_transform)
val_dataset   = RetinalDataset(val_df,   transform=val_transform)

# create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, 
                          shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=16, 
                          shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

# test one batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Labels: {labels}")

Train batches: 184
Val batches:   46

Batch shape: torch.Size([16, 3, 300, 300])
Labels: tensor([2, 2, 1, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0])


In [4]:
class RetinalClassifier(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3):
        super(RetinalClassifier, self).__init__()
        
        # load pretrained EfficientNet-B3
        self.backbone = timm.create_model(
            'efficientnet_b3',
            pretrained=True,
            num_classes=0  # remove original classifier
        )
        
        # get feature size
        feature_size = self.backbone.num_features
        
        # new classification head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

# create model
model = RetinalClassifier(num_classes=5).to(device)

# count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"\nModel ready on {device}")

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

C:\Users\hidey\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hidey\.cache\huggingface\hub\models--timm--efficientnet_b3.ra2_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Total parameters:     11,090,989
Trainable parameters: 11,090,989

Model ready on cpu


In [6]:
# compute class weights to handle imbalance
class_counts = train_df['diagnosis'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * 5  # normalize
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print("Class weights:")
grade_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
for i, (name, w) in enumerate(zip(grade_names, class_weights)):
    print(f"  Grade {i} ({name}): {w:.4f}")

# loss with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# optimizer
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

print(f"Loss: CrossEntropyLoss with class weights")
print(f"Optimizer: Adam lr=0.0001")
print(f"Scheduler: ReduceLROnPlateau patience=3")

Class weights:
  Grade 0 (No DR): 0.2157
  Grade 1 (Mild): 1.0522
  Grade 2 (Moderate): 0.3898
  Grade 3 (Severe): 2.0225
  Grade 4 (Proliferative): 1.3198
Loss: CrossEntropyLoss with class weights
Optimizer: Adam lr=0.0001
Scheduler: ReduceLROnPlateau patience=3


In [2]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in tqdm(loader, desc='Training', leave=False):
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = running_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds, 
                              weights='quadratic')
    return avg_loss, accuracy, kappa

def val_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation', leave=False):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = running_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds,
                              weights='quadratic')
    return avg_loss, accuracy, kappa

# training history
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [],  'val_acc': [],
    'train_kappa': [], 'val_kappa': []
}

best_kappa = 0.0
epochs = 10

print("Starting training...")
print(f"{'Epoch':>5} {'T-Loss':>8} {'T-Acc':>7} {'T-Kappa':>9} "
      f"{'V-Loss':>8} {'V-Acc':>7} {'V-Kappa':>9}")
print("-" * 60)

for epoch in range(epochs):
    t_loss, t_acc, t_kappa = train_epoch(
        model, train_loader, criterion, optimizer)
    v_loss, v_acc, v_kappa = val_epoch(
        model, val_loader, criterion)
    
    scheduler.step(v_loss)
    
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    history['train_kappa'].append(t_kappa)
    history['val_kappa'].append(v_kappa)
    
    # save best model
    if v_kappa > best_kappa:
        best_kappa = v_kappa
        torch.save(model.state_dict(), '../data/best_model.pth')
        flag = ' ← best'
    else:
        flag = ''
    
    print(f"{epoch+1:>5} {t_loss:>8.4f} {t_acc:>7.3f} {t_kappa:>9.4f} "
          f"{v_loss:>8.4f} {v_acc:>7.3f} {v_kappa:>9.4f}{flag}")

print(f"\nTraining complete. Best validation kappa: {best_kappa:.4f}")

Starting training...
Epoch   T-Loss   T-Acc   T-Kappa   V-Loss   V-Acc   V-Kappa
------------------------------------------------------------


NameError: name 'model' is not defined